# Simpson's Paradox Investigation

Investigating why individual-level effects (fast contact → higher conversion) disappear at the location level.

**Three analyses:**
1. Control for lead characteristics by location
2. Within-location analysis: fast vs slow contact conversion rates
3. Lead mix comparison across high vs low converting locations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# Load data (INCLUDING UNUSED_IND=1 records)
file_path = '../data/raw/Conversion Data Nov-Dec 2025 (1).xlsx'
df = pd.read_excel(file_path, engine='openpyxl')

# Clean column names
df.columns = df.columns.str.strip()

# Note: Including ALL records (UNUSED_IND=0 and UNUSED_IND=1)
unused_count = df['UNUSED_IND'].sum() if 'UNUSED_IND' in df.columns else 0
print(f"Total records: {len(df):,}")
print(f"  - UNUSED_IND=1: {unused_count:,} ({unused_count/len(df)*100:.1f}%)")
print(f"  - UNUSED_IND=0: {len(df)-unused_count:,} ({(len(df)-unused_count)/len(df)*100:.1f}%)")
print(f"\nColumns available: {list(df.columns)}")

In [ ]:
# Quick look at key columns
print("Data types:")
print(df.dtypes)
print("\n" + "="*50)
print("\nSample data:")
df.head()

---
## Analysis 1: Control for Lead Characteristics by Location

Check if certain locations receive systematically different types of leads.

In [ ]:
# Identify categorical columns that might represent lead characteristics
potential_lead_chars = []
for col in df.columns:
    if df[col].dtype == 'object' or df[col].nunique() < 20:
        if col not in ['RENT_LOC', 'RES_ID', 'RENT_IND', 'CONTACT RANGE', 'CONTACT_GROUP']:
            potential_lead_chars.append(col)

print("Potential lead characteristic columns:")
for col in potential_lead_chars:
    print(f"  {col}: {df[col].nunique()} unique values")

In [ ]:
# Calculate location-level metrics
location_stats = df.groupby('RENT_LOC').agg(
    total_reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum'),
    total_records=('RES_ID', 'count')
).reset_index()

location_stats['conversion_rate'] = (location_stats['conversions'] / location_stats['total_reservations'] * 100)

# Filter to locations with meaningful volume
location_stats = location_stats[location_stats['total_reservations'] >= 50].copy()

# Classify locations as high or low converting
median_conv = location_stats['conversion_rate'].median()
location_stats['conv_group'] = np.where(location_stats['conversion_rate'] >= median_conv, 'High Converting', 'Low Converting')

print(f"Median conversion rate: {median_conv:.1f}%")
print(f"High converting locations: {(location_stats['conv_group'] == 'High Converting').sum()}")
print(f"Low converting locations: {(location_stats['conv_group'] == 'Low Converting').sum()}")

In [ ]:
# Merge location classification back to main data
df_with_loc_class = df.merge(
    location_stats[['RENT_LOC', 'conv_group', 'conversion_rate']], 
    on='RENT_LOC', 
    how='inner'
)

print(f"Records after merge: {len(df_with_loc_class):,}")

In [ ]:
# Compare lead characteristics between high and low converting locations
print("="*60)
print("LEAD CHARACTERISTICS: High vs Low Converting Locations")
print("="*60)

# Check contact speed distribution
print("\n1. CONTACT SPEED DISTRIBUTION:")
contact_by_group = pd.crosstab(
    df_with_loc_class['conv_group'], 
    df_with_loc_class['CONTACT RANGE'], 
    normalize='index'
) * 100
print(contact_by_group.round(1))

print("\n2. CONTACT GROUP DISTRIBUTION:")
contact_group_by_conv = pd.crosstab(
    df_with_loc_class['conv_group'], 
    df_with_loc_class['CONTACT_GROUP'], 
    normalize='index'
) * 100
print(contact_group_by_conv.round(1))

In [ ]:
# Check other lead characteristics if available
char_cols_to_check = [col for col in potential_lead_chars if col in df.columns and df[col].nunique() <= 10]

for col in char_cols_to_check[:5]:  # Check top 5 categorical variables
    print(f"\n{col.upper()} DISTRIBUTION:")
    try:
        crosstab = pd.crosstab(
            df_with_loc_class['conv_group'], 
            df_with_loc_class[col], 
            normalize='index'
        ) * 100
        print(crosstab.round(1))
    except:
        print(f"  Could not analyze {col}")

---
## Analysis 2: Within-Location Analysis

For each location, compare conversion rates of fast vs slow contacted leads. This controls for location-level confounders.

In [ ]:
# Create fast contact indicator
df['fast_contact'] = (df['CONTACT RANGE'] == '(a)<30min').astype(int)
df['counter_contact'] = (df['CONTACT_GROUP'] == 'COUNTER').astype(int)

# Calculate within-location conversion rates by contact speed
within_loc_analysis = df.groupby(['RENT_LOC', 'fast_contact']).agg(
    reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum')
).reset_index()

within_loc_analysis['conv_rate'] = within_loc_analysis['conversions'] / within_loc_analysis['reservations'] * 100

# Pivot to compare fast vs slow within each location
within_loc_pivot = within_loc_analysis.pivot(
    index='RENT_LOC', 
    columns='fast_contact', 
    values=['conv_rate', 'reservations']
)

within_loc_pivot.columns = ['conv_slow', 'conv_fast', 'n_slow', 'n_fast']
within_loc_pivot = within_loc_pivot.dropna()

# Filter to locations with at least 10 in each group
within_loc_pivot = within_loc_pivot[(within_loc_pivot['n_slow'] >= 10) & (within_loc_pivot['n_fast'] >= 10)].copy()

# Calculate the lift from fast contact
within_loc_pivot['fast_contact_lift'] = within_loc_pivot['conv_fast'] - within_loc_pivot['conv_slow']

print(f"Locations with sufficient data in both groups: {len(within_loc_pivot)}")

In [ ]:
# Summary statistics of within-location lift
print("WITHIN-LOCATION FAST CONTACT LIFT (percentage points):")
print(f"  Mean lift: {within_loc_pivot['fast_contact_lift'].mean():.1f} pp")
print(f"  Median lift: {within_loc_pivot['fast_contact_lift'].median():.1f} pp")
print(f"  Std dev: {within_loc_pivot['fast_contact_lift'].std():.1f} pp")
print(f"  Min: {within_loc_pivot['fast_contact_lift'].min():.1f} pp")
print(f"  Max: {within_loc_pivot['fast_contact_lift'].max():.1f} pp")
print(f"\n  Locations where fast contact helps: {(within_loc_pivot['fast_contact_lift'] > 0).sum()} ({(within_loc_pivot['fast_contact_lift'] > 0).mean()*100:.0f}%)")
print(f"  Locations where fast contact hurts: {(within_loc_pivot['fast_contact_lift'] < 0).sum()} ({(within_loc_pivot['fast_contact_lift'] < 0).mean()*100:.0f}%)")

In [ ]:
# Visualize the within-location lift distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of lift
axes[0].hist(within_loc_pivot['fast_contact_lift'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No effect')
axes[0].axvline(x=within_loc_pivot['fast_contact_lift'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {within_loc_pivot["fast_contact_lift"].mean():.1f} pp')
axes[0].set_xlabel('Fast Contact Lift (percentage points)', fontsize=12)
axes[0].set_ylabel('Number of Locations', fontsize=12)
axes[0].set_title('Distribution of Within-Location Fast Contact Effect', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: slow vs fast conversion rates
axes[1].scatter(within_loc_pivot['conv_slow'], within_loc_pivot['conv_fast'], alpha=0.5)
axes[1].plot([0, 100], [0, 100], 'r--', label='No difference line')
axes[1].set_xlabel('Conversion Rate - Slow Contact (%)', fontsize=12)
axes[1].set_ylabel('Conversion Rate - Fast Contact (%)', fontsize=12)
axes[1].set_title('Fast vs Slow Contact Conversion by Location', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Same analysis for Counter contact
within_loc_counter = df.groupby(['RENT_LOC', 'counter_contact']).agg(
    reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum')
).reset_index()

within_loc_counter['conv_rate'] = within_loc_counter['conversions'] / within_loc_counter['reservations'] * 100

within_loc_counter_pivot = within_loc_counter.pivot(
    index='RENT_LOC', 
    columns='counter_contact', 
    values=['conv_rate', 'reservations']
)

within_loc_counter_pivot.columns = ['conv_non_counter', 'conv_counter', 'n_non_counter', 'n_counter']
within_loc_counter_pivot = within_loc_counter_pivot.dropna()
within_loc_counter_pivot = within_loc_counter_pivot[(within_loc_counter_pivot['n_non_counter'] >= 10) & (within_loc_counter_pivot['n_counter'] >= 10)].copy()
within_loc_counter_pivot['counter_lift'] = within_loc_counter_pivot['conv_counter'] - within_loc_counter_pivot['conv_non_counter']

print("WITHIN-LOCATION COUNTER CONTACT LIFT (percentage points):")
print(f"  Mean lift: {within_loc_counter_pivot['counter_lift'].mean():.1f} pp")
print(f"  Median lift: {within_loc_counter_pivot['counter_lift'].median():.1f} pp")
print(f"  Locations where counter helps: {(within_loc_counter_pivot['counter_lift'] > 0).sum()} ({(within_loc_counter_pivot['counter_lift'] > 0).mean()*100:.0f}%)")

---
## Analysis 3: Lead Mix Comparison

Compare the composition of leads at high-converting vs low-converting locations.

In [ ]:
# Aggregate lead mix metrics by location
location_mix = df.groupby('RENT_LOC').agg(
    total_reservations=('RES_ID', 'sum'),
    conversions=('RENT_IND', 'sum'),
    pct_fast_contact=('fast_contact', 'mean'),
    pct_counter=('counter_contact', 'mean'),
    pct_no_contact=('CONTACT RANGE', lambda x: (x == 'NO CONTACT').mean())
).reset_index()

location_mix['conversion_rate'] = location_mix['conversions'] / location_mix['total_reservations'] * 100
location_mix['pct_fast_contact'] *= 100
location_mix['pct_counter'] *= 100
location_mix['pct_no_contact'] *= 100

# Filter to meaningful volume
location_mix = location_mix[location_mix['total_reservations'] >= 50].copy()

# Classify
median_conv = location_mix['conversion_rate'].median()
location_mix['conv_group'] = np.where(location_mix['conversion_rate'] >= median_conv, 'High Converting', 'Low Converting')

location_mix.head()

In [ ]:
# Compare metrics between high and low converting locations
comparison = location_mix.groupby('conv_group').agg({
    'conversion_rate': ['mean', 'std'],
    'pct_fast_contact': ['mean', 'std'],
    'pct_counter': ['mean', 'std'],
    'pct_no_contact': ['mean', 'std'],
    'total_reservations': ['mean', 'sum']
}).round(1)

print("COMPARISON: High vs Low Converting Locations")
print("="*60)
print(comparison)

In [ ]:
# Statistical test: Do high-converting locations have different contact patterns?
from scipy import stats

high_conv = location_mix[location_mix['conv_group'] == 'High Converting']
low_conv = location_mix[location_mix['conv_group'] == 'Low Converting']

print("T-TEST RESULTS: High vs Low Converting Locations")
print("="*60)

for metric in ['pct_fast_contact', 'pct_counter', 'pct_no_contact']:
    t_stat, p_val = stats.ttest_ind(high_conv[metric], low_conv[metric])
    high_mean = high_conv[metric].mean()
    low_mean = low_conv[metric].mean()
    diff = high_mean - low_mean
    sig = "*" if p_val < 0.05 else ""
    print(f"\n{metric}:")
    print(f"  High conv locations: {high_mean:.1f}%")
    print(f"  Low conv locations:  {low_mean:.1f}%")
    print(f"  Difference: {diff:+.1f} pp (p={p_val:.3f}) {sig}")

In [ ]:
# Visualize the paradox
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Location-level correlation (no relationship)
axes[0].scatter(location_mix['pct_fast_contact'], location_mix['conversion_rate'], alpha=0.5)
z = np.polyfit(location_mix['pct_fast_contact'], location_mix['conversion_rate'], 1)
p = np.poly1d(z)
x_line = np.linspace(location_mix['pct_fast_contact'].min(), location_mix['pct_fast_contact'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', alpha=0.7)
corr = location_mix['pct_fast_contact'].corr(location_mix['conversion_rate'])
axes[0].set_xlabel('% Fast Contact at Location')
axes[0].set_ylabel('Location Conversion Rate (%)')
axes[0].set_title(f'Location-Level: r = {corr:.2f}\n(No correlation)')
axes[0].grid(True, alpha=0.3)

# Plot 2: Within-location effect (strong relationship)
axes[1].hist(within_loc_pivot['fast_contact_lift'], bins=25, edgecolor='black', alpha=0.7, color='green')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].axvline(x=within_loc_pivot['fast_contact_lift'].mean(), color='darkgreen', linestyle='-', linewidth=2)
axes[1].set_xlabel('Conversion Lift from Fast Contact (pp)')
axes[1].set_ylabel('Number of Locations')
axes[1].set_title(f'Within-Location Effect\nMean lift: {within_loc_pivot["fast_contact_lift"].mean():.1f} pp')
axes[1].grid(True, alpha=0.3)

# Plot 3: Box plot comparison
data_to_plot = [high_conv['pct_fast_contact'], low_conv['pct_fast_contact']]
bp = axes[2].boxplot(data_to_plot, labels=['High Converting', 'Low Converting'], patch_artist=True)
bp['boxes'][0].set_facecolor('green')
bp['boxes'][1].set_facecolor('red')
axes[2].set_ylabel('% Fast Contact')
axes[2].set_title('Fast Contact Rate by Location Type\n(Similar distributions)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary & Key Findings

In [ ]:
print("="*70)
print("SIMPSON'S PARADOX INVESTIGATION: SUMMARY")
print("="*70)

print("\n1. INDIVIDUAL-LEVEL EFFECT (Within-Location Analysis):")
print(f"   - Fast contact provides {within_loc_pivot['fast_contact_lift'].mean():.1f} pp lift on average")
print(f"   - Effect is positive in {(within_loc_pivot['fast_contact_lift'] > 0).mean()*100:.0f}% of locations")
print(f"   - Counter contact provides {within_loc_counter_pivot['counter_lift'].mean():.1f} pp lift on average")

print("\n2. LOCATION-LEVEL CORRELATION:")
corr_fast = location_mix['pct_fast_contact'].corr(location_mix['conversion_rate'])
corr_counter = location_mix['pct_counter'].corr(location_mix['conversion_rate'])
print(f"   - % Fast Contact vs Conversion Rate: r = {corr_fast:.3f}")
print(f"   - % Counter vs Conversion Rate: r = {corr_counter:.3f}")

print("\n3. WHY THE PARADOX EXISTS:")
print("   - High and low converting locations have SIMILAR contact patterns")
print("   - Location conversion is driven by OTHER factors (lead quality, market, etc.)")
print("   - Fast contact helps WITHIN each location, but doesn't explain")
print("     BETWEEN-location differences")

print("\n4. IMPLICATION:")
print("   - Contact speed DOES matter for conversion (individual effect is real)")
print("   - But it's not the PRIMARY driver of location-level differences")
print("   - Other factors (unmeasured) explain why some locations convert better")